# Pump Telemetry Snippets

Small patterns you can copy into the participant workbook if you get stuck. Keep the pieces you need and adjust the column names to match the task.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

csv_candidates = [Path('pump_telemetry.csv'), Path.cwd() / 'pump_telemetry.csv']
for candidate in csv_candidates:
    if candidate.exists():
        csv_path = candidate
        break
else:
    raise FileNotFoundError('pump_telemetry.csv was not found in the current folder.')

telemetry_df = pd.read_csv(csv_path, parse_dates=['timestamp'])
print(telemetry_df.head())
print(telemetry_df.info())

In [ ]:
working_df = telemetry_df.copy()
working_df[['raw_vibration', 'drifted_pressure', 'motor_temp', 'fluid_temp']] = working_df[['raw_vibration', 'drifted_pressure', 'motor_temp', 'fluid_temp']].ffill()
working_df['cleaned_vibration'] = savgol_filter(working_df['raw_vibration'].to_numpy(), window_length=15, polyorder=2)
working_df['calibrated_pressure'] = [pressure + 0.05 * hour for hour, pressure in zip(working_df['time_hours'], working_df['drifted_pressure'])]


In [ ]:
def rolling_rms(signal, window=61):
    series = pd.Series(signal, dtype='float64')
    return (
        series.rolling(window=window, min_periods=1, center=True)
        .apply(lambda values: float(np.sqrt(np.mean(np.square(values)))), raw=True)
        .bfill()
        .ffill()
        .to_numpy()
    )

def rolling_trend(signal, window=31):
    values = np.asarray(signal, dtype=float)
    trend_values = []
    half_window = window // 2
    for index in range(values.shape[0]):
        start = max(0, index - half_window)
        stop = min(values.shape[0], index + half_window + 1)
        window_values = values[start:stop]
        trend_values.append((window_values[-1] - window_values[0]) / max(1, window_values.size - 1))
    return pd.Series(trend_values).bfill().ffill().to_numpy()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
import shap

feature_cols = ['vibration_rms', 'vibration_trend', 'pressure_reserve', 'motor_temp']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
predictions = model.predict(X_test)
print(r2_score(y_test, predictions))

# Uncertainty pattern from tree spread
tree_predictions = np.stack([tree.predict(X_test) for tree in model.estimators_], axis=0)
prediction_mean = tree_predictions.mean(axis=0)
prediction_std = tree_predictions.std(axis=0)

# SHAP pattern
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test.iloc[:50] if hasattr(X_test, 'iloc') else X_test[:50])